In [ ]:
# =============================================================================
# 0. SETUP
# =============================================================================
import math
from datetime import datetime, timezone
 
import ee
import pandas as pd
 
ee.Authenticate()
ee.Initialize(project='osmgee')

In [19]:
ASSET_FOLDER = 'projects/annual-ccdc-new/assets'
PREFIX = 'medoid_composite_'

assets = ee.data.listAssets({'parent': ASSET_FOLDER, 'pageSize': 1000})['assets']
composite_ids = sorted(a['id'] for a in assets
                       if a['type'] == 'IMAGE' and a['id'].split('/')[-1].startswith(PREFIX))

print('Number of composites:', len(composite_ids))
for cid in composite_ids:
    print(cid.split('/')[-1])

# Inspect one of them
img = ee.Image(composite_ids[0])
print('\nBands:', img.bandNames().getInfo())
print('Data types:', img.bandTypes().getInfo())
proj = img.select(0).projection().getInfo()
print('CRS:', proj['crs'], '| Pixel size:', img.select(0).projection().nominalScale().getInfo(), 'm')

Number of composites: 41
medoid_composite_1985
medoid_composite_1986
medoid_composite_1987
medoid_composite_1988
medoid_composite_1989
medoid_composite_1990
medoid_composite_1991
medoid_composite_1992
medoid_composite_1993
medoid_composite_1994
medoid_composite_1995
medoid_composite_1996
medoid_composite_1997
medoid_composite_1998
medoid_composite_1999
medoid_composite_2000
medoid_composite_2001
medoid_composite_2002
medoid_composite_2003
medoid_composite_2004
medoid_composite_2005
medoid_composite_2006
medoid_composite_2007
medoid_composite_2008
medoid_composite_2009
medoid_composite_2010
medoid_composite_2011
medoid_composite_2012
medoid_composite_2013
medoid_composite_2014
medoid_composite_2015
medoid_composite_2016
medoid_composite_2017
medoid_composite_2018
medoid_composite_2019
medoid_composite_2020
medoid_composite_2021
medoid_composite_2022
medoid_composite_2023
medoid_composite_2024
medoid_composite_2025

Bands: ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
Data types: {'B

In [20]:
img = ee.Image(composite_ids[0])   # 1985

stats = img.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.percentile([1, 99]), sharedInputs=True),
    geometry=img.geometry().bounds(),
    scale=300, maxPixels=1e10, bestEffort=True, tileScale=4)

for k, v in sorted(stats.getInfo().items()):
    print(f'{k:15s} {v}')

Blue_max        0.10124617187500001
Blue_min        0.009236499999999995
Blue_p1         0.015407108852488293
Blue_p99        0.05300284724443022
Green_max       0.15654076562500002
Green_min       0.013520106653620339
Green_p1        0.028777715144798002
Green_p99       0.09814773961653137
NIR_max         0.3928402734375
NIR_min         0.011559218749999987
NIR_p1          0.01475981997282608
NIR_p99         0.30964097351561387
Red_max         0.19682517187499998
Red_min         0.0112528515625
Red_p1          0.02002753158798533
Red_p99         0.07669406933270502
SWIR1_max       0.36401100186704893
SWIR1_min       0.00042988281249999467
SWIR1_p1        0.0015444606711647693
SWIR1_p99       0.21967247621713426
SWIR2_max       0.31043540625000005
SWIR2_min       -0.0013675000000000052
SWIR2_p1        0.0009303383608217617
SWIR2_p99       0.15724206876445132


In [21]:
DRIVE_FOLDER_COMP = 'AOSR_Medoid_Composites'
CRS, SCALE = 'EPSG:32612', 30
NODATA = -32768

def export_composite(asset_id):
    year = asset_id.split('_')[-1]
    img = ee.Image(asset_id)
    out = (img.multiply(10000).round().toInt16()
              .unmask(NODATA))
    fname = f'medoid_composite_{year}'
    ee.batch.Export.image.toDrive(
        image=out,
        description=fname,
        folder=DRIVE_FOLDER_COMP,
        fileNamePrefix=fname,
        region=img.geometry(),
        scale=SCALE,
        crs=CRS,
        maxPixels=1e13,
        fileFormat='GeoTIFF',
        formatOptions={'noData': NODATA},
    ).start()
    print('Started:', fname)

export_composite(composite_ids[0])   # 1985 only

Started: medoid_composite_1985


In [22]:
done = {t.config.get('description') for t in ee.batch.Task.list()
        if t.state in ('READY', 'RUNNING', 'COMPLETED')}

for cid in composite_ids[1:]:          # 1986–2025
    fname = 'medoid_composite_' + cid.split('_')[-1]
    if fname in done:
        print('Skipped (already done/queued):', fname)
        continue
    export_composite(cid)

Started: medoid_composite_1986
Started: medoid_composite_1987
Started: medoid_composite_1988
Started: medoid_composite_1989
Started: medoid_composite_1990
Started: medoid_composite_1991
Started: medoid_composite_1992
Started: medoid_composite_1993
Started: medoid_composite_1994
Started: medoid_composite_1995
Started: medoid_composite_1996
Started: medoid_composite_1997
Started: medoid_composite_1998
Started: medoid_composite_1999
Started: medoid_composite_2000
Started: medoid_composite_2001
Started: medoid_composite_2002
Started: medoid_composite_2003
Started: medoid_composite_2004
Started: medoid_composite_2005
Started: medoid_composite_2006
Started: medoid_composite_2007
Started: medoid_composite_2008
Started: medoid_composite_2009
Started: medoid_composite_2010
Started: medoid_composite_2011
Started: medoid_composite_2012
Started: medoid_composite_2013
Started: medoid_composite_2014
Started: medoid_composite_2015
Started: medoid_composite_2016
Started: medoid_composite_2017
Started:

In [23]:
task_summary()

CANCELLED    201
READY         37
RUNNING        3
COMPLETED      2
